In [ ]:
# ================== SEL PERSIAPAN A - Muat library & file dataset ==================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

NAMA_FILE = "Seduh Coffee - Final Project (BNSP) - Sept 2026 v02.xlsx"

# Di Google Colab: kalau file belum ada, kotak upload akan muncul.
if not os.path.exists(NAMA_FILE):
    try:
        from google.colab import files
        print("Silakan pilih file:", NAMA_FILE)
        diupload = files.upload()
        NAMA_FILE = list(diupload.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"'{NAMA_FILE}' tidak ditemukan. Letakkan file di folder yang sama dengan notebook ini."
        )

print("File siap dipakai:", NAMA_FILE)

# Palet warna kelas (mengikuti tema materi)
WARNA = {
    "tosca":  "#4BD4B1",
    "gelap":  "#0B3B36",
    "sedang": "#12615A",
    "muda":   "#BFE6DC",
    "oranye": "#E8833A",
    "merah":  "#C4553B",
    "abu":    "#7A8C89",
}
URUTAN_SEGMEN = [
    "Champions", "Loyal Customers", "Potential Loyalist", "Need Attention",
    "At Risk", "Can't Lose Them", "Hibernating",
]
WARNA_SEGMEN = {
    "Champions":          "#0B3B36",
    "Loyal Customers":    "#12615A",
    "Potential Loyalist": "#4BD4B1",
    "Need Attention":     "#BFE6DC",
    "At Risk":            "#E8833A",
    "Can't Lose Them":    "#C4553B",
    "Hibernating":        "#7A8C89",
}

def rp(x):
    """Format angka menjadi Rupiah singkat, mis. 3_008_035_000 -> 'Rp 3,01 M'."""
    x = float(x)
    if abs(x) >= 1_000_000_000:
        return f"Rp {x/1_000_000_000:,.2f} M".replace(",", "#").replace(".", ",").replace("#", ".")
    if abs(x) >= 1_000_000:
        return f"Rp {x/1_000_000:,.1f} jt".replace(",", "#").replace(".", ",").replace("#", ".")
    return f"Rp {x:,.0f}".replace(",", ".")

print("Sel Persiapan A selesai.")

In [ ]:
# ================== SEL PERSIAPAN B - Baca & rapikan data (sudah disiapkan) ==================
# Tidak ada yang perlu diubah di sel ini. Jalankan saja.
# Isinya adalah langkah pembersihan standar yang sudah kita pelajari di sesi sebelumnya,
# supaya waktu kelas hari ini bisa fokus penuh ke RFM & segmentasi.

orders_raw = pd.read_excel(NAMA_FILE, sheet_name="Orders")
customers  = pd.read_excel(NAMA_FILE, sheet_name="Customers")
products   = pd.read_excel(NAMA_FILE, sheet_name="Products")

# Nama channel di data mentah tidak konsisten: kapitalisasi, spasi depan/belakang, dan salah ketik.
RAPIKAN_CHANNEL = {
    "Tokopedia": "Tokopedia", "Tokopdia": "Tokopedia",
    "Shopee":    "Shopee",    "Shoppee":  "Shopee",
    "Webstore":  "Webstore",
    "Tiktok Shop": "TikTok Shop",
}

orders = (
    orders_raw
      .drop_duplicates()                                   # buang baris kembar persis
      .drop_duplicates(subset="order_id", keep="first")    # 1 order_id = 1 baris
      .assign(
          # sebagian order_date tersimpan sebagai TEKS "dd/mm/yyyy" -> dayfirst=True
          order_date=lambda d: pd.to_datetime(d["order_date"], dayfirst=True, errors="coerce"),
          quantity=lambda d: pd.to_numeric(d["quantity"], errors="coerce"),
          channel=lambda d: d["channel"].str.strip().str.title().map(RAPIKAN_CHANNEL),
          city=lambda d: d["city"].str.strip().str.title(),
      )
      .query("quantity > 0")                               # buang qty negatif / nol
      .query("order_status == 'Completed'")                # RFM hanya memakai transaksi sah
      .copy()
)

# Kolom turunan: nilai transaksi setelah diskon
orders["revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount_pct"])

# Rapikan tabel Customers (untuk profiling & ekspor daftar target di akhir sesi)
RAPIKAN_GENDER = {
    "Female": "Female", "F": "Female", "female": "Female", "FEMALE": "Female", "Perempuan": "Female",
    "Male": "Male", "M": "Male", "male": "Male", "MALE": "Male", "Laki-laki": "Male",
}
customers = customers.assign(
    gender=lambda d: d["gender"].map(RAPIKAN_GENDER),
    customer_name=lambda d: d["customer_name"].str.strip()
                             .str.replace(r"\s+", " ", regex=True).str.title(),
    city=lambda d: d["city"].fillna("Tidak diketahui").str.strip().str.title(),
)

print(f"Baris order mentah    : {len(orders_raw):,}")
print(f"Baris order siap pakai: {len(orders):,}")
print(f"Rentang tanggal       : {orders['order_date'].min():%d %b %Y} s/d {orders['order_date'].max():%d %b %Y}")
print("Sel Persiapan B selesai.")

In [ ]:
# 1a. Ukuran tiap tabel
print("orders   :", orders.shape)
print("customers:", customers.shape)
print("products :", products.shape)

# 1b. Lihat isinya
display(orders.head())

# 1c. Berapa pelanggan yang pernah bertransaksi?
print("\nPelanggan unik di orders   :", orders["customer_id"].nunique())
print("Pelanggan terdaftar         :", customers["customer_id"].nunique())
print("Belum pernah belanja        :",
      customers["customer_id"].nunique() - orders["customer_id"].nunique())

In [ ]:
# TODO 2a-2d: lengkapi lima angka ringkasan di bawah
total_revenue = ____
jumlah_order  = ____
jumlah_pelanggan = ____
aov = ____

print("Total revenue    :", rp(total_revenue))
print("Jumlah order     :", f"{jumlah_order:,}")
print("Jumlah pelanggan :", f"{jumlah_pelanggan:,}")
print("AOV              :", rp(aov))

# TODO 2e: revenue per channel + kontribusi persen + AOV per channel
per_channel = (
    orders.groupby("____")
          .agg(revenue=("____", "____"),
               order=("____", "____"))
)
per_channel["kontribusi_%"] = ____
per_channel["aov"] = ____
display(per_channel.sort_values("revenue", ascending=False).round(1))

In [ ]:
# 3a. Tentukan snapshot date
snapshot_date = orders["order_date"].max() + pd.Timedelta(days=1)
print("Transaksi terakhir :", orders["order_date"].max().strftime("%d %b %Y"))
print("Snapshot date      :", snapshot_date.strftime("%d %b %Y"))

# 3b. Tanggal transaksi TERAKHIR tiap pelanggan
terakhir = orders.groupby("customer_id")["order_date"].max()

# 3c. Recency = selisih hari terhadap snapshot date
recency = (snapshot_date - terakhir).dt.days
recency.name = "Recency"

display(recency.head())
print("\nRecency  min:", recency.min(), "hari | median:", int(recency.median()),
      "hari | max:", recency.max(), "hari")

In [ ]:
# TODO 4: bangun tabel rfm lengkap dalam SATU groupby().agg()
rfm = (
    orders.groupby("customer_id")
          .agg(
              Recency   = ("order_date", lambda s: ____),
              Frequency = ("____", "____"),
              Monetary  = ("____", "____"),
          )
          .reset_index()
)

print("Jumlah baris rfm:", len(rfm), "(harus sama dengan jumlah pelanggan unik)")
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(1))

In [ ]:
# 5a. Skor Recency - perhatikan urutan label yang DIBALIK
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# 5b. Cek batas tiap kuintil supaya skornya bisa dijelaskan ke bisnis
batas_r = rfm.groupby("R_Score")["Recency"].agg(["min", "max", "count"])
batas_r.index.name = "R_Score"
display(batas_r)

print("\nTerjemahan bisnisnya:")
for skor in [5, 4, 3, 2, 1]:
    b = batas_r.loc[skor]
    print(f"  R_Score {skor}: belanja terakhir {int(b['min'])}-{int(b['max'])} hari lalu "
          f"({int(b['count'])} pelanggan)")

In [ ]:
# TODO 6a: F_Score - kuintil akan gagal di sini. Coba dulu baris berikut supaya
#          teman-teman melihat sendiri error "Bin edges must be unique":
# rfm["F_Score"] = pd.qcut(rfm["Frequency"], q=5, labels=[1, 2, 3, 4, 5])

def skor_frequency(jumlah_order):
    """Ambang bisnis: >=6 -> 5, 4-5 -> 4, 3 -> 3, 2 -> 2, 1 -> 1."""
    if jumlah_order >= 6:
        return 5
    # ... lanjutkan ...
    return ____

rfm["F_Score"] = rfm["Frequency"].apply(____)

# TODO 6b: M_Score - Monetary kontinu, qcut aman
rfm["M_Score"] = pd.qcut(____, q=5, labels=[____]).astype(int)

# TODO 6c: skor gabungan (angka) dan pola sel (teks)
rfm["RFM_Score"] = ____
rfm["RFM_Cell"]  = ____

display(rfm.head())

# TODO 6d: 5 pelanggan terbaik
display(rfm.sort_values("____", ascending=False).head(5))

# Cek sebaran F_Score
print("\nSebaran F_Score:")
print(rfm.groupby("F_Score")["Frequency"].agg(["min", "max", "count"]))

In [ ]:
def segmentasi_demo(baris):
    """Versi awal - baru 3 aturan. Kita lengkapi di latihan berikutnya."""
    r, f, m = baris["R_Score"], baris["F_Score"], baris["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:        # paling spesifik -> paling atas
        return "Champions"
    if r >= 3 and f >= 3:
        return "Loyal Customers"
    if r <= 2 and f <= 2:
        return "Hibernating"
    return "Belum terklasifikasi"

rfm["Segment_Demo"] = rfm.apply(segmentasi_demo, axis=1)
display(rfm["Segment_Demo"].value_counts().to_frame("jumlah_pelanggan"))

sisa = (rfm["Segment_Demo"] == "Belum terklasifikasi").sum()
print(f"\nMasih ada {sisa:,} pelanggan ({sisa/len(rfm):.1%}) yang belum punya nama segmen.")

In [ ]:
def segmentasi(baris):
    """Terjemahkan R/F/M Score menjadi nama segmen. Urutan aturan menentukan hasil!"""
    r, f, m = baris["R_Score"], baris["F_Score"], baris["M_Score"]

    # TODO 8a: susun 7 aturan di sini, dari yang paling spesifik ke paling umum
    if ____:
        return "Champions"
    # ... lanjutkan ...

    return "Need Attention"   # penampung terakhir

rfm["Segment"] = rfm.apply(segmentasi, axis=1)

# TODO 8b: validasi - tidak boleh ada yang kosong / tak dikenal
assert rfm["Segment"].notna().all(), "Masih ada pelanggan tanpa segmen!"
assert rfm["Segment"].nunique() == 7, "Jumlah segmen belum 7"

# TODO 8c: jumlah + persentase per segmen
ringkas_segmen = ____
display(ringkas_segmen)

In [ ]:
jumlah = rfm["Segment"].value_counts().reindex(URUTAN_SEGMEN)

fig, ax = plt.subplots(figsize=(10, 5.2))
bars = ax.barh(
    jumlah.index[::-1], jumlah.values[::-1],
    color=[WARNA_SEGMEN[s] for s in jumlah.index[::-1]],
)

# Label nilai langsung di ujung bar - lebih mudah dibaca daripada sumbu X
for bar, nilai in zip(bars, jumlah.values[::-1]):
    ax.text(bar.get_width() + 12, bar.get_y() + bar.get_height() / 2,
            f"{nilai:,}  ({nilai/len(rfm):.0%})",
            va="center", fontsize=10, color="#123331")

ax.set_title("Jumlah Pelanggan per Segmen RFM", fontsize=15, fontweight="bold",
             color="#0B3B36", pad=14)
ax.set_xlabel("Jumlah pelanggan")
ax.set_xlim(0, jumlah.max() * 1.22)
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
for sisi in ["top", "right", "left"]:
    ax.spines[sisi].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ---------- (a) Pareto: % pelanggan vs % revenue ----------
# TODO 10a: bangun tabel bantu berisi persen pelanggan & persen revenue per segmen
pareto = (
    rfm.groupby("Segment")
       .agg(pelanggan=("customer_id", "____"),
            revenue=("Monetary", "____"))
)
pareto["pct_pelanggan"] = ____
pareto["pct_revenue"]   = ____
pareto = pareto.sort_values("pct_revenue")

y = np.arange(len(pareto))
tinggi = 0.38
# TODO 10b: dua set bar berdampingan (gunakan y - tinggi/2 dan y + tinggi/2)
ax1.barh(____, ____, height=tinggi, label="% pelanggan", color=WARNA["muda"])
ax1.barh(____, ____, height=tinggi, label="% revenue",   color=WARNA["gelap"])
ax1.set_yticks(y); ax1.set_yticklabels(pareto.index)
ax1.set_title("Porsi Pelanggan vs Porsi Revenue", fontsize=14, fontweight="bold", color="#0B3B36")
ax1.set_xlabel("Persen (%)"); ax1.legend(frameon=False, loc="lower right")
for sisi in ["top", "right", "left"]:
    ax1.spines[sisi].set_visible(False)

# ---------- (b) Heatmap R x F ----------
# TODO 10c: tabel silang jumlah pelanggan per kombinasi R_Score x F_Score
matriks = pd.crosstab(____, ____)
matriks = matriks.sort_index(ascending=False)   # R_Score 5 di baris atas

gambar = ax2.imshow(matriks.values, cmap="BuGn", aspect="auto")
ax2.set_xticks(range(len(matriks.columns))); ax2.set_xticklabels(matriks.columns)
ax2.set_yticks(range(len(matriks.index)));   ax2.set_yticklabels(matriks.index)
ax2.set_xlabel("F_Score  (makin ke kanan makin sering belanja)")
ax2.set_ylabel("R_Score  (makin ke atas makin baru belanja)")
ax2.set_title("Peta Sebaran Pelanggan: Recency x Frequency",
              fontsize=14, fontweight="bold", color="#0B3B36")

# TODO 10d: tulis angka di setiap kotak heatmap
for i in range(matriks.shape[0]):
    for j in range(matriks.shape[1]):
        nilai = ____
        ax2.text(j, i, f"{nilai}", ha="center", va="center",
                 color="white" if nilai > matriks.values.max() * 0.55 else "#0B3B36",
                 fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
profil = (
    rfm.groupby("Segment")
       .agg(
           jumlah_pelanggan = ("customer_id", "count"),
           avg_recency      = ("Recency", "mean"),
           avg_frequency    = ("Frequency", "mean"),
           avg_monetary     = ("Monetary", "mean"),
           total_revenue    = ("Monetary", "sum"),
       )
       .sort_values("total_revenue", ascending=False)
)

profil["pct_pelanggan"]   = profil["jumlah_pelanggan"] / profil["jumlah_pelanggan"].sum() * 100
profil["pct_revenue"]     = profil["total_revenue"]    / profil["total_revenue"].sum()    * 100
profil["revenue_per_org"] = profil["total_revenue"]    / profil["jumlah_pelanggan"]

tampil = profil.copy()
tampil["avg_recency"]     = tampil["avg_recency"].round(0).astype(int).astype(str) + " hari"
tampil["avg_frequency"]   = tampil["avg_frequency"].round(1)
tampil["avg_monetary"]    = tampil["avg_monetary"].apply(rp)
tampil["total_revenue"]   = tampil["total_revenue"].apply(rp)
tampil["revenue_per_org"] = tampil["revenue_per_org"].apply(rp)
tampil["pct_pelanggan"]   = tampil["pct_pelanggan"].round(1)
tampil["pct_revenue"]     = tampil["pct_revenue"].round(1)

display(tampil)

In [ ]:
# TODO 12a: revenue at risk
segmen_berisiko = ["At Risk", "Can't Lose Them"]
revenue_at_risk = ____
pct_at_risk     = ____
print("Revenue at risk :", rp(revenue_at_risk), f"({pct_at_risk:.1f}% dari total)")

# TODO 12b: potensi upside dari Potential Loyalist (target: order KEDUA)
pl = rfm[rfm["Segment"] == "Potential Loyalist"]
nilai_per_order   = ____          # rata-rata Monetary dibagi rata-rata Frequency
frekuensi_target  = 2
tambahan_revenue  = ____
print("Potensi tambahan revenue:", rp(tambahan_revenue))

# TODO 12c: ekspor daftar target kampanye
target = (
    rfm[rfm["Segment"] == "____"]
      .merge(customers[["customer_id", "customer_name", "city"]], on="____", how="left")
      [["customer_id", "customer_name", "city", "Recency", "Frequency", "Monetary"]]
      .sort_values("____", ascending=False)
)
target.to_csv("target_winback_cant_lose_them.csv", index=False)
print("File tersimpan. Jumlah target:", len(target))
display(target.head(10))